# Canopy — Eyeball the layers

The Master Guide's one rule: **always look before you trust a number.** This notebook
renders every layer on an interactive map for one of the two demo projects. All numbers
come from the `canopy_pipeline` modules (single source of truth) — this notebook only *draws*.

> Run `python -c "import ee; ee.Authenticate()"` once in the terminal before the first run.

## 0 · Auth & pick a project

In [ ]:
import ee, geemap
from canopy_pipeline import init_ee
from canopy_pipeline.examples import EXAMPLES
from canopy_pipeline.input.aoi_handler import build_aoi, area_hectares
from canopy_pipeline.data.sentinel2_loader import build_composite
from canopy_pipeline.data.sentinel1_loader import water_layer
from canopy_pipeline.indices.ndvi import ndvi
from canopy_pipeline.indices.ndwi import ndwi
from canopy_pipeline.analytics.change_detection import detect

init_ee()

# choose: 'mikoko' (mangrove) or 'nur_navoi' (solar)
REQUEST = EXAMPLES['mikoko']
aoi = build_aoi(REQUEST)
print(REQUEST['project_name'], '|', round(area_hectares(aoi), 1), 'ha')

## 1 · Before / after RGB
Should look like the real place. Blank/weird -> widen dates or raise cloud_pct.

In [ ]:
before_img, n_before = build_composite(aoi, REQUEST['before'], REQUEST['cloud_pct'])
after_img,  n_after  = build_composite(aoi, REQUEST['after'],  REQUEST['cloud_pct'])

m = geemap.Map()
m.centerObject(aoi, 14)
rgb = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}
m.addLayer(before_img, rgb, 'before RGB')
m.addLayer(after_img,  rgb, 'after RGB')
m.addLayer(aoi, {}, 'AOI')
m

## 2 · NDVI, NDWI (water) & change map
NDVI green = vegetation. NDWI blue = water (key for the mangrove tidal edge).
Change: **red = loss, green = gain.**

In [ ]:
nd_before, nd_after = ndvi(before_img), ndvi(after_img)
diff, change, veg_change_pct = detect(nd_before, nd_after, aoi)

m.addLayer(nd_after, {'min': 0, 'max': 1, 'palette': ['white', 'green']}, 'NDVI after')
m.addLayer(ndwi(after_img), {'min': -0.3, 'max': 0.5, 'palette': ['white', 'blue']}, 'NDWI water')
m.addLayer(diff, {'min': -0.3, 'max': 0.3, 'palette': ['red', 'white', 'green']}, 'change')
print('vegetation change %:', round(veg_change_pct, 1))
m

## 3 · SAR water layer (Sentinel-1, feeds flood risk)

In [ ]:
water, n_sar = water_layer(aoi, REQUEST['after'], optical_img=after_img)
print('SAR scenes:', n_sar)
m.addLayer(water.selfMask(), {'palette': ['00b3ff']}, 'SAR water')
m

## 4 · The numbers — full pipeline result (spec §3 JSON)

In [ ]:
import json
from canopy_pipeline import run_pipeline
print(json.dumps(run_pipeline(REQUEST), indent=2, default=str))